# Stereo-seq (FF footpad tumors): Lrrc15⁺ fibroblasts (Fig. 3D–F; Supplementary Fig. S9)
- **Fig. 3D** spatial distribution of Lrrc15⁺ fibroblasts and their proportion across layers
- **Supplementary Fig. S9** Lrrc15⁺ fibroblasts in the remaining biological replicates
- **Fig. 3E** Acta2 / Lrrc15 expression across tumor regions (box plots)
- **Fig. 3F** differentially expressed genes of peritumor fibroblasts and enriched pathways

In [ ]:
import os
import re
import importlib.metadata
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import anndata as ad
import decoupler as dc
from scipy import stats

from spatial_utils import *

%matplotlib inline
plt.rcParams['pdf.fonttype'] = 42
sc.settings.verbosity = 3
sc.set_figure_params(dpi = 100, dpi_save = 300, transparent=True, color_map = 'OrRd')
os.makedirs('./figures', exist_ok=True)

for package in ['stereopy', 'numpy', 'pandas', 'matplotlib', 'scanpy', 'anndata', 'squidpy', 'decoupler']:
    try:
        print(f"{package} version: {importlib.metadata.version(package)}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package} is not installed")

# Annotated bin30 objects: <GEO library name>_bin30_annotated.h5ad
SPATIAL_DIR = '../data/stereoseq_annotated'
sample_list = SAMPLE_LIST

In [ ]:
adataL = load_margin_lognorm(SPATIAL_DIR, sample_list)
lognorm4 = load_scored(SPATIAL_DIR, sample_list)

## Fig. 3D — Spatial distribution of Lrrc15⁺ fibroblasts (representative slides)

In [ ]:
layers_to_view = ['-150um~-100um', '-100um~-50um', '-50um~0um', '0um~50um', '50um~100um', '100um~150um', '150um~200um', '200um~250um', '250um~300um', '300um~350um', '350um~400um', '400um~450um', '450um~500um', '500um~550um', '550um~600um', '600um~650um', '650um~700um', '700um~750um', '750um~800um', '800um~850um', '850um~900um', '900um~950um', '950um~1000um', '1000um~1050um', '1050um~1100um', '1100um~1150um', '1150um~1200um', '1200um~1250um', '1250um~1300um', '1300um~1350um', '1350um~1400um', '1400um~1450um', '1450um~1500um']

In [ ]:
# Colours of the manuscript panels: Background, Border, Lrrc15+ fibroblast (FB_Lrrc15+)
for sample in sample_list:
    lognorm4[sample].uns['Lrrc15_border_colors'] = ['#DBDBDB', '#E53522', '#3A5AE6']
    adataL[sample].uns['Lrrc15_border_colors'] = ['#DBDBDB', '#E53522', '#3A5AE6']

In [ ]:
for sample in ['FF_Mattress_1', 'FF_Running_3', 'FF_YAP5SA_2']:
    bdata = adataL[sample]
    bdata = bdata[bdata.obs['tumor_layer'].isin(layers_to_view)]
    sc.pl.spatial(bdata, color = 'Lrrc15_border' , spot_size =50, frameon = False, title = f'{sample}', save=f'{sample}_Lrrc15.pdf')

## Supplementary Fig. S9 — Remaining biological replicates

In [ ]:
sc.set_figure_params(dpi=1200, color_map = 'OrRd', facecolor=None, transparent=True)

In [ ]:
for sample in sample_list:
    if sample not in ['FF_Mattress_1', 'FF_Running_3', 'FF_YAP5SA_2']:
        bdata = adataL[sample]
        bdata = bdata[bdata.obs['tumor_layer'].isin(layers_to_view)]
        bdata.uns['Lrrc15_border_colors'] = ['#DBDBDB', '#E53522', '#3A5AE6']
        sc.pl.spatial(bdata, color = 'Lrrc15_border', spot_size=50, frameon=False, title = f'{sample}')

In [ ]:
sc.set_figure_params(dpi=300, color_map = 'OrRd', facecolor=None, transparent=True)

## Fig. 3D — Proportion of Lrrc15⁺ fibroblasts across layers

In [ ]:
proportion_dict = {}
for sample in sample_list:
    df = pd.crosstab(adataL[sample].obs['cell_type_level2'], adataL[sample].obs['tumor_layer'])
    df = df.div(df.sum(axis=0), axis = 1)
    proportion_dict[sample] = df
    del df

In [ ]:
plot_cell_type_proportions_line(proportion_dict, cell_types='FB_Lrrc15+')

## Fig. 3E — Acta2 and Lrrc15 expression across tumor regions
Mean log-normalised expression of all bins in each 50 µm layer, averaged over the layers of each region
(peritumor -150–0 µm, margin 0–150 µm, center 150–700 µm); one value per sample. Statistics: one-way ANOVA over all
region × group combinations with Tukey's post hoc test (only comparisons within a region are listed).

In [ ]:
for gene in ['Acta2', 'Lrrc15']:
    box_df = region_mean_expression(lognorm4, gene)
    display(box_df.pivot(index='Sample', columns='Region', values='Value')[list(REGION_LAYERS)])
    display(tukey_region_group(box_df))
    fig, ax = plot_region_box(box_df, gene)
    plt.show()

## Fig. 3F — Differentially expressed genes of peritumor fibroblasts
Pseudobulk (summed counts of fibroblast bins in the peritumor layers, per sample) and PyDESeq2 for each pair of groups
(padj < 0.05). The heatmap shows the union of the DEGs, grouped by the group in which they are highest
(group means of the log-normalised pseudobulk; z-score difference > 0.5, as in the original analysis), and Enrichr
(gseapy) is run on the genes high in Running and high in YAP5SA.

The DESeq2 cell below re-runs the test; the figure uses the DEG tables of the original analysis
(`../data/Peritumor_Fibroblast_DEG_*_pseudo.csv`), because PyDESeq2 versions differ slightly in the dispersion
estimates and therefore in the genes close to padj = 0.05.

In [ ]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

combined_adata = ad.concat(lognorm4)
peritumor = REGION_LAYERS['Peritumor']
CONTRASTS = {'M_vs_R': ['group', 'Running', 'Mattress'],
             'R_vs_Y': ['group', 'YAP5SA', 'Running'],
             'M_vs_Y': ['group', 'YAP5SA', 'Mattress']}
deg_rerun = {}
for name, contrast in CONTRASTS.items():
    idx = ((combined_adata.obs['cell_type'] == 'Fibroblast') & combined_adata.obs['tumor_layer'].isin(peritumor)
           & combined_adata.obs['group'].isin(contrast[1:]))
    pdata = dc.get_pseudobulk(combined_adata[idx].copy(), sample_col='sample', groups_col='cell_type',
                              layer='counts', mode='sum', min_cells=10, min_counts=1000)
    dds = DeseqDataSet(adata=pdata, design_factors='group', refit_cooks=True, min_mu=0.1, quiet=True)
    dds.deseq2()
    stat_res = DeseqStats(dds, contrast=contrast, quiet=True)
    stat_res.summary()
    res = stat_res.results_df.dropna()
    deg_rerun[name] = res[res['padj'] < 0.05].sort_values('log2FoldChange', ascending=False)
    deg_rerun[name].to_csv(f'./figures/Peritumor_Fibroblast_DEG_{name}_pseudo_rerun.csv')

In [ ]:
deg = {name: pd.read_csv(f'../data/Peritumor_Fibroblast_DEG_{name}_pseudo.csv', index_col=0) for name in CONTRASTS}
for name in deg:
    shared = deg[name].index.intersection(deg_rerun[name].index)
    print(f'{name}: {len(deg[name])} DEGs (original analysis), {len(deg_rerun[name])} re-run, {len(shared)} shared')
deg_genes = sorted(set().union(*[set(d.index) for d in deg.values()]))

# log-normalised pseudobulk of peritumor fibroblasts, all samples
idx = (combined_adata.obs['cell_type'] == 'Fibroblast') & combined_adata.obs['tumor_layer'].isin(peritumor)
pdata = dc.get_pseudobulk(combined_adata[idx].copy(), sample_col='sample', groups_col='cell_type',
                          layer='counts', mode='sum', min_cells=10, min_counts=1000)
sc.pp.normalize_total(pdata, target_sum=1e4)
sc.pp.log1p(pdata)
pdata.obs_names = pdata.obs['sample'].astype(str)
genes = [g for g in deg_genes if g in pdata.var_names]
X = pdata[sample_list, genes].X
expr = pd.DataFrame(X.toarray() if hasattr(X, 'toarray') else X, index=sample_list, columns=genes).T

groups = ['Mattress', 'Running', 'YAP5SA']
means = pd.DataFrame({g: expr[[s for s in sample_list if extract_group(s) == g]].mean(axis=1) for g in groups})


def classify_pattern(row, threshold=0.5):
    z = (row.values - row.values.mean()) / row.values.std()
    i = np.argmax(z)
    if z[i] > threshold and all(z[i] - z[j] > threshold for j in range(3) if j != i):
        return f'High in {groups[i]}'
    o = np.argsort(z)
    if z[o[2]] > threshold and z[o[1]] > threshold and z[o[2]] - z[o[0]] > threshold and z[o[1]] - z[o[0]] > threshold:
        return 'High in ' + ' & '.join(sorted([groups[o[2]], groups[o[1]]]))
    return 'Other'


pattern = means.apply(classify_pattern, axis=1)
print(pattern.value_counts())

# Heatmap: genes high in one group, grouped Mattress / Running / YAP5SA
blocks = {g: pattern.index[pattern == f'High in {g}'] for g in groups}
zmeans = means.sub(means.mean(axis=1), axis=0)
order = sum([list(zmeans.loc[blocks[g], g].sort_values(ascending=False).index) for g in groups], [])
fig, ax = plot_group_zscore_heatmap(expr.loc[order], [extract_group(s) for s in sample_list])
fig.set_label('Peritumor_fibroblast_DEG_heatmap')
plt.show()

In [ ]:
import gseapy as gp

LIBRARIES = ['GO_Biological_Process_2023', 'GO_Molecular_Function_2023', 'KEGG_2019_Mouse', 'MSigDB_Hallmark_2020',
             'Reactome_2022', 'WikiPathways_2024_Mouse']
# Terms shown in the manuscript (Enrichr term -> label)
PANEL_TERMS = {
    'Running': {'Epithelial Mesenchymal Transition': 'Epithelial Mesenchymal Transition',
                'Extracellular Matrix Organization R-HSA-1474244': 'Extracellular matrix Organization (R)',
                'Extracellular Matrix Organization (GO:0030198)': 'Extracellular matrix Organization (GO)',
                'Focal Adhesion WP85': 'Focal Adhesion (WP)',
                'Assembly Of Collagen Fibrils And Other Multimeric Structures R-HSA-2022090':
                    'Assembly Of Collagen Fibrils&Other Multimeric Structures',
                'Collagen Formation R-HSA-1474290': 'Collagen Formation',
                'ECM-receptor interaction': 'ECM-receptor interaction',
                'Focal adhesion': 'Focal adhesion (KEGG)',
                'Collagen Fibril Organization (GO:0030199)': 'Collagen Fibril Organization',
                'Collagen Biosynthesis And Modifying Enzymes R-HSA-1650814': 'Collagen Biosynthesis And Modifying Enzymes'},
    'YAP5SA': {'RNA Binding (GO:0003723)': 'RNA Binding',
               'Metabolism Of RNA R-HSA-8953854': 'Metabolism Of RNA',
               'Interferon Gamma Response': 'Interferon Gamma Response',
               'Myc Targets V1': 'Myc Targets V1',
               'Interferon Alpha Response': 'Interferon Alpha Response',
               'KEAP1-NFE2L2 Pathway R-HSA-9755511': 'KEAP1-NFE2L2 Pathway',
               'Nuclear Events Mediated By NFE2L2 R-HSA-9759194': 'Nuclear Events Mediated By NFE2L2',
               'Proteasome': 'Proteasome',
               'Cell Cycle R-HSA-1640170': 'Cell Cycle',
               'G1/S DNA Damage Checkpoints R-HSA-69615': 'G1/S DNA Damage Checkpoints'}}

for g in ['Running', 'YAP5SA']:
    enr = gp.enrichr(gene_list=list(blocks[g]), gene_sets=LIBRARIES, organism='Mouse', outdir=None).results
    enr = enr.sort_values('Adjusted P-value')
    enr.to_csv(f'./figures/Fig3F_enrichr_High_in_{g}.csv', index=False)
    display(enr.head(12)[['Gene_set', 'Term', 'Overlap', 'Adjusted P-value']])
    sel = enr.drop_duplicates('Term').set_index('Term').reindex(list(PANEL_TERMS[g]))
    fig, ax = plot_hbar(-np.log10(sel['Adjusted P-value'].values), list(PANEL_TERMS[g].values()),
                        color=GROUP_COLORS[g], xlabel='-log(Adjusted P-value)', title=f'{g} high',
                        figsize=(3, 3), top_axis=False)
    fig.set_label(f'{g}_high_enrichr')
    plt.show()